In [ ]:
pip install seaborn

In [ ]:
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from pathlib import Path

In [ ]:
RAW_DATA_DIR = Path("../data/raw")
PROCESSED_DATA_DIR = Path("../data/processed")

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
tickers = {
    "BTC": "BTC-USD",
    "ETH": "ETH-USD",
    "NASDAQ": "^NDX",
    "SP500": "^GSPC",
    "GOLD": "GC=F",
    "DXY": "DX-Y.NYB",
    "US10Y": "^TNX"
}

In [ ]:
start_date = "2018-01-01"
end_date = None

data = yf.download(
    list(tickers.values()),
    start=start_date,
    end=end_date,
    auto_adjust=True,
    progress=False
)

data.head()

In [ ]:
close_prices = data["Close"].copy()

# Rename columns from ticker symbols to readable asset names
reverse_tickers = {v: k for k, v in tickers.items()}
close_prices = close_prices.rename(columns=reverse_tickers)

close_prices.head()

In [ ]:
missing_summary = close_prices.isna().sum().sort_values(ascending=False)
missing_summary

In [ ]:
# Forward fill missing values
close_prices_clean = close_prices.ffill()

# Remove rows that still contain any missing values
close_prices_clean = close_prices_clean.dropna()

close_prices_clean.head()

In [ ]:
print("Shape:", close_prices_clean.shape)
print("Start date:", close_prices_clean.index.min())
print("End date:", close_prices_clean.index.max())

close_prices_clean.tail()

In [ ]:
output_path = PROCESSED_DATA_DIR / "multi_asset_prices.csv"

close_prices_clean.to_csv(output_path)

print(f"Saved to: {output_path}")

In [ ]:
check_df = pd.read_csv(output_path)

check_df.head()

In [ ]:
prices = pd.read_csv(
    "../data/processed/multi_asset_prices.csv",
    index_col=0,
    parse_dates=True
)

prices.head()

In [ ]:
normalized_prices = prices / prices.iloc[0] * 100

normalized_prices.head()

In [ ]:
plt.figure(figsize=(14, 7))

for column in normalized_prices.columns:
    plt.plot(
        normalized_prices.index,
        normalized_prices[column],
        label=column
    )

plt.title("Normalized Asset Performance Since 2018")
plt.xlabel("Date")
plt.ylabel("Normalized Performance (Base = 100)")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
returns = close_prices_clean.pct_change().dropna()

returns.head()

In [ ]:
summary_stats = returns.describe().T

summary_stats

In [ ]:
annualized_volatility = returns.std() * np.sqrt(252)

annualized_volatility.sort_values(ascending=False)

In [ ]:
annualized_volatility.sort_values().plot(
    kind="barh",
    figsize=(10, 5)
)

plt.title("Annualized Volatility Comparison")
plt.xlabel("Volatility")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
correlation_matrix = returns.corr()

correlation_matrix